# 手写self-attention的四个境界

## step1 了解公式+计算图

![alt text](resource/self-attention_cal_graph.png)

![alt text](resource/attention2.png)

![alt text](resource/attention_cal.png)

# step2 代码

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

## 第一重：简化版本

In [47]:
class selfAttentionV1(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.emb_dim = emb_dim
        self.query_proj = nn.Linear(emb_dim, emb_dim)
        self.key_proj = nn.Linear(emb_dim, emb_dim)
        self.value_proj = nn.Linear(emb_dim, emb_dim)
    def forward(self, X):
        # query, key, value shape:(batch_size, seq_len, emb_dim)
        Q = self.query_proj(X)
        K = self.key_proj(X)
        V = self.value_proj(X)
        
        # attention_score shape:(batch_size, seq_len, seq_len)
        attention_score = torch.matmul(
            # 转置K-> shape(batch_size, emb_dim, seq_len)
            Q, K.transpose(-1,-2)# 交换后两维度
        )
        
        
        attention_scaled_score = attention_score / math.sqrt(self.emb_dim)
        # 提问：为什么要除这个：attention_score的方差为d_k，容易出现超大值，后续进入softmax会输出近似于ont-hot，导致导致反向传播时梯度几乎消失
        # a. 防止梯度消失 b. 为了让 QK 的内积分布保持和输入一样
        
        # attention_weight shape:(batch_size, seq_len, seq_len)
        attention_weight = F.softmax(attention_scaled_score, dim=-1)
        #F.softmax(dim=-1):softmax独立地作用于张量在最后一个维度上的每一个“切片”，通常都为-1
        
        # output shape:(batch_size, seq_len, emb_dim)
        output = torch.matmul(attention_weight, V)
        return output
    
X = torch.rand(3, 2, 4)
net = selfAttentionV1(4)
net(X).shape

torch.Size([3, 2, 4])

## 第二重：效率优化
- 上面那哪些操作可以合并矩阵优化呢？- QKV 矩阵计算的时候，可以合并成一个大矩阵计算。
- 但是当前 transformers 实现中，其实是三个不同的 Linear 层

In [48]:
class selfAttentionV2(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.emb_dim = emb_dim
        self.proj = nn.Linear(emb_dim, 3*emb_dim)
    def forward(self, X):
        # query, key, value shape:(batch_size, seq_len, emb_dim)
        QKV = self.proj(X)
        Q, K, V = torch.split(QKV, self.emb_dim, dim=-1)
        # 将倒数第一维度按emb_dim长度拆分
        
        # attention_score shape:(batch_size, seq_len, seq_len)
        # 转置K-> shape(batch_size, emb_dim, seq_len)
        attention_score = Q @ K.transpose(-1,-2)# 交换后两维度
        
        
        attention_scaled_score = attention_score / math.sqrt(self.emb_dim)
        # 提问：为什么要除这个：attention_score的方差为d_k，容易出现超大值，后续进入softmax会输出近似于ont-hot，导致导致反向传播时梯度几乎消失
        
        # attention_weight shape:(batch_size, seq_len, seq_len)
        attention_weight = F.softmax(attention_scaled_score, dim=-1)
        #F.softmax(dim=-1):softmax独立地作用于张量在最后一个维度上的每一个“切片”，通常都为-1
        
        # output shape:(batch_size, seq_len, emb_dim)
        output = attention_weight @ V
        return output
    
X = torch.rand(3, 2, 4)
net = selfAttentionV2(4)
net(X).shape

torch.Size([3, 2, 4])

## 第三重：添加细节
- 1.dropout位置 :attention 计算的时候有 dropout，而且是比较奇怪的位置
- 2.attention_mask :attention 计算的时候一般会加入 attention_mask，因为样本会进行一些 padding 操作；
- 3.output 矩阵映射

In [49]:
class selfAttentionV3(nn.Module):
    def __init__(self, dim, droupout_rate=0.1) -> None:
        super().__init__()
        self.dim = dim
        
        self.proj = nn.Linear(dim, 3*dim)
        self.attention_dropout = nn.Dropout(droupout_rate)
        self.output_proj = nn.Linear(dim, dim)
        
    def forward(self, X, attention_mask=None):
        # X shape:(batch_size, seq_len, dim)
        QKV = self.proj(X)
        Q, K, V = torch.split(QKV, self.dim, dim=-1)
        # 将倒数第一维度按dim长度拆分
        
        # attention_score shape:(batch_size, seq_len, seq_len)
        # 转置K-> shape(batch_size, dim, seq_len)
        attention_score = Q @ K.transpose(-1,-2)# 交换后两维度
        
        attention_scaled_score = attention_score / math.sqrt(self.dim)
        # 提问：为什么要除这个：attention_score的方差为d_k，容易出现超大值，后续进入softmax会输出近似于ont-hot，导致导致反向传播时梯度几乎消失
        
        if attention_mask is not None:
            attention_scaled_score = attention_scaled_score.masked_fill(
                attention_mask==0,
                float('-inf')
            )
            #tensor.masked_fill(bool的矩阵, 填充数值)，会在tensor中bool为True的位置填充数值，因为后续过softmax，故取-inf
        
        # attention_weight shape:(batch_size, seq_len, seq_len)
        attention_weight = F.softmax(attention_scaled_score, dim=-1)
        
        attention_weight = self.attention_dropout(attention_weight)
        
        # output shape:(batch_size, seq_len, dim)
        attention_result = attention_weight @ V
        output = self.output_proj(attention_result)
        return output
    
X = torch.rand(3, 4, 2)
b = torch.tensor(
    [
        [1, 1, 1, 0],
        [1, 1, 0, 0],
        [1, 0, 0, 0],
    ]
)
print(b.shape)
# mask shape:(batch_size, seq_len, seq_len) = (3,4,4)而不是与X相同
mask = b.unsqueeze(dim=1).repeat(1, 4, 1)
'''
repeated_tensor = tensor.repeat(*sizes)
tensor: 输入的原始张量。
*sizes: 一个整数序列，指定在每个维度上重复的次数。你可以传入多个参数，也可以传入一个元组或列表。
'''
net = selfAttentionV3(2)
net(X, mask).shape

torch.Size([3, 4])


torch.Size([3, 4, 2])

## 面试写法 （完整版）--注意注释

In [50]:
class SelfAttentionInterView(nn.Module):
    def __init__(self, dim, dropout_rate=0.1):
        super().__init__()
        self.dim = dim
        
        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)
        
        self.attention_dropout = nn.Dropout(dropout_rate)
        
        self.output_proj = nn.Linear(dim, dim)
    def forward(self, X, attention_mask=None):
        # X shape: (batch_size, seq_len, dim)
        
        # Q,K,V shape like X
        Q = self.query(X)
        K = self.key(X)
        V = self.value(X)
        print(f'Q:{Q.shape}')
        print(f'K:{K.shape}')
        print(f'V:{V.shape}')
        
        # attention_scores shape: (batch_size, seq_len, seq_len)
        attention_scores = Q @ K.transpose(-1,-2)
        print(f'attention_scores:{attention_scores.shape}')
        
        # attention_scaled_scores shape like attention_scores
        attention_scaled_scores = attention_scores / math.sqrt(self.dim)
        print(f'attention_scaled_scores:{attention_scaled_scores.shape}')
        
        # att_mask like scaled_att_scores: (b,s,s)
        if attention_mask is not None:
            attention_scaled_scores = attention_scaled_scores.masked_fill(
                attention_mask == 0, float('-inf')
            )# tensor.masked_fill(尺寸相同的bool矩阵, 填充的值)
        
        # attention_weights shape like attention_scores
        attention_weights = F.softmax(attention_scaled_scores, dim=-1)
        print(f'attention_weights:{attention_weights.shape}')
        
        # attention_result shape :(batch_size, seq_len, dim)
        attention_result = attention_weights @ V
        print(f'attention_result:{attention_result.shape}')
        
        output = self.output_proj(attention_result)
        print(f'output:{output.shape}')
        return output
        

batch_size = 3
seq_len = 4
dim = 2
X = torch.randn(batch_size,seq_len,dim)
b = torch.tensor(
    [
    [1,1,1,0],
    [1,0,0,0],
    [1,1,0,0],
    ]
)
print(b.shape)
# mask shape:(batch_size, seq_len, seq_len) = (3,4,4)而不是与X相同
mask = b.unsqueeze(dim=1).repeat(1, 4, 1)
'''
repeated_tensor = tensor.repeat(*sizes)
tensor: 输入的原始张量。
*sizes: 一个整数序列，指定在每个维度上重复的次数。你可以传入多个参数，也可以传入一个元组或列表。
tensor.unsqueeze(dim):在维度dim插入一个新维度，用来扩宽维度
'''
net = SelfAttentionInterView(dim)
net(X, mask).shape

torch.Size([3, 4])
Q:torch.Size([3, 4, 2])
K:torch.Size([3, 4, 2])
V:torch.Size([3, 4, 2])
attention_scores:torch.Size([3, 4, 4])
attention_scaled_scores:torch.Size([3, 4, 4])
attention_weights:torch.Size([3, 4, 4])
attention_result:torch.Size([3, 4, 2])
output:torch.Size([3, 4, 2])


torch.Size([3, 4, 2])

## 第四重 手写多头注意力机制（multi-head_attention）

![alt text](resource/multi-head-attention.png)


![alt text](resource/multi-head-attention_cal.png)

In [44]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, hidden_dim, head_num, attention_dropout=0.1):
        super(MultiHeadSelfAttention, self).__init__()

        self.head_num = head_num
        self.head_dim = hidden_dim // head_num # (head_num * head_dim = hidden_dim)
        # 为了将多个(head_num)头concat到一起的时候，保持dim=hidden_dim
        self.q_proj = nn.Linear(hidden_dim, hidden_dim) # hidden_dim, head_dim * head_num
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.output_proj = nn.Linear(hidden_dim, hidden_dim)
        
        self.attention_dropout = nn.Dropout(attention_dropout)
        
    def forward(self, X, attention_mask=None):
        
        batch, seq_len, _ = X.size()
        print(f'batch:{batch}')
        print(f'seq_len:{seq_len}')
        print(f'head_num:{self.head_num}')
        print(f'head_dim:{self.head_dim}')
        
        # X shape:(batch_size, seq, hidden_dim)
        Q = self.q_proj(X)
        K = self.k_proj(X)
        V = self.v_proj(X) 
        # Q,K,V shape :(batch_size, seq, hidden_dim)
        print(f'Q:{Q.shape}')
        print(f'K:{K.shape}')
        print(f'V:{V.shape}')
        
        # 拆分 (batch_size, seq, hidden_dim) -> (batch_size, head_num, seq, head_dim) , h->head_num * head_dim
        # q_state, k_state, v_state : (batch_size, head_num, seq, head_dim)
        q_state = Q.reshape(batch, seq_len, self.head_num, self.head_dim).transpose(1,2)
        k_state = K.reshape(batch, seq_len, self.head_num, self.head_dim).transpose(1,2)
        v_state = V.reshape(batch, seq_len, self.head_num, self.head_dim).transpose(1,2)
        # tensor.view(**shape) 几乎等价于tensor.reshape(**shape)，都是重新定义维度dim
        print(f'q_state:{q_state.shape}')
        print(f'k_state:{k_state.shape}')
        print(f'k_state:{k_state.shape}')
        
        # attention_weight shape:(batch_size, head_num, seq, seq)
        attention_weight = torch.matmul(
            # 将k_state(batch_size, head_num, seq, head_dim)->(batch_size, head_num, head_dim, seq)
            q_state, k_state.transpose(-1,-2)
        ) / math.sqrt(self.head_dim)
        
        if attention_mask is not None:
            attention_weight = attention_weight.masked_fill(
                attention_mask == 0, float('-inf')
            )
            
        # attention_weight shape:(batch_size, head_num, seq, seq)
        attention_weight = F.softmax(attention_weight, dim=-1)
        
        # attention_weight shape:(batch_size, head_num, seq, seq)
        attention_weight = self.attention_dropout(attention_weight)
        print(f'attention_weight:{attention_weight.shape}')
        
        # output_mid : (batch_size, head_num, seq, head_dim)
        output_mid = torch.matmul(
            attention_weight, v_state
        )
        print(f'output_mid:{output_mid.shape}')
        
        output_mid = output_mid.transpose(1,2).contiguous()
        print(f'output_mid:{output_mid.shape}')
        
        output_mid = output_mid.reshape(batch,seq_len, -1)
        # tensor.contiguous()可以获得内存连续的新张量，使用view会让内存不连续，报错
        print(f'output_mid:{output_mid.shape}')
        
        output = self.output_proj(output_mid)
        print(f'output:{output.shape}')
        return output
        

In [45]:
attention_mask = (
    torch.tensor(
        [
            [0, 1],
            [0, 0],
            [1, 0],
        ]
    )
    .unsqueeze(1)
    .unsqueeze(2)
    .expand(3, 8, 2, 2)
)

x = torch.rand(3, 2, 128)
net = MultiHeadSelfAttention(128, 8)
net(x, attention_mask).shape

batch:3
seq_len:2
head_num:8
head_dim:16
Q:torch.Size([3, 2, 128])
K:torch.Size([3, 2, 128])
V:torch.Size([3, 2, 128])
q_state:torch.Size([3, 8, 2, 16])
k_state:torch.Size([3, 8, 2, 16])
k_state:torch.Size([3, 8, 2, 16])
attention_weight:torch.Size([3, 8, 2, 2])
output_mid:torch.Size([3, 8, 2, 16])
output_mid:torch.Size([3, 2, 8, 16])
output_mid:torch.Size([3, 2, 128])
output:torch.Size([3, 2, 128])


torch.Size([3, 2, 128])